# 06 - Why the final model behaves as it does, and where it errs

Notebook 05 showed large recall gaps between groups of positives. This notebook asks where they come from, which accounts the model gets wrong, and whether the same accounts are wrong under other fold assignments. Nothing here changes the model, so test scores are diagnostics, not selection. Flags use the current rule, best F1 on the validation fold; the rank columns do not depend on any threshold. Only aggregates are shown.

In [1]:
%%time
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)

from skeptical_second_reader.data import (
    ACCOUNTS_CSV,
    PATTERNS_TXT,
    account_entities,
    account_key,
    account_labels,
    load_transactions,
    pattern_members,
    read_accounts,
    read_patterns,
)
from skeptical_second_reader.features import ALL_FEATURES, WINDOW_END
from skeptical_second_reader.model import fit_final
from skeptical_second_reader.paths import DATA_INTERIM
from skeptical_second_reader.split import (
    N_FOLDS,
    cached_folds,
    cv_predict,
    split_groups,
)

df = load_transactions()
labels = account_labels(df)
owners = account_entities(labels.index, read_accounts(ACCOUNTS_CSV))
members = pattern_members(read_patterns(PATTERNS_TXT))
groups = split_groups(owners, members)
folds = cached_folds(labels, groups, DATA_INTERIM)

X = pd.read_parquet(DATA_INTERIM / "experiment_features.parquet")[ALL_FEATURES]
results = pd.read_parquet(DATA_INTERIM / "final_scores.parquet")
explained = pd.read_parquet(DATA_INTERIM / "final_shap.parquet")
assert X.index.equals(results.index) and X.index.equals(explained.index)
assert results["label"].equals(labels.loc[X.index].rename("label"))
assert results["fold"].equals(folds.loc[X.index].rename("fold"))

y = results["label"]
flags = results["flag"]
scores = results["score"]
fold_of = results["fold"]
ranks = scores.rank(pct=True)

sender = account_key(df["from_bank"], df["from_account"])
receiver = account_key(df["to_bank"], df["to_account"])
in_window = df["timestamp"] < WINDOW_END
is_self = sender == receiver
bad = in_window & (df["is_laundering"] == 1)
is_ach = df["payment_format"] == "ACH"


def touches(mask):
    """Transactions per account within `mask`; a self-transaction counts once."""
    counted = pd.concat([sender[mask], receiver[mask & ~is_self]])
    return counted.value_counts().reindex(X.index, fill_value=0)


total_tx = touches(in_window)
bad_count = touches(bad)
ach_bad_count = touches(bad & is_ach)
any_ach = (X["ach_out_tx"] + X["ach_in_tx"]) > 0
in_pattern = pd.Series(X.index.isin(members["account"]), index=X.index)

positive = y == 1
negative = y == 0
has_ach_label = ach_bad_count > 0
non_ach_label = (bad_count > 0) & ~has_ach_label
tail_only = positive & (bad_count == 0)
print(f"accounts {len(X):,}  positives {positive.sum():,}  flagged {flags.sum():,}")
print(
    f"positives: ACH label {(positive & has_ach_label).sum():,}, non-ACH label "
    f"{(positive & non_ach_label).sum():,}, tail only {tail_only.sum():,}"
)

accounts 515,078  positives 6,347  flagged 2,383
positives: ACH label 5,122, non-ACH label 740, tail only 485
CPU times: total: 17 s
Wall time: 14.6 s


## 1. Is the ACH gap only a matter of how many labelled transactions an account has?

First the profile of each group, then recall within strata that hold the suspected confounders fixed. `round_min` and `round_max` are the lowest and highest recall over the five test folds. `top_1pct` is the share of the group among the top 1% of all scores.

In [2]:
def recall_row(mask):
    rows = mask & positive
    by_round = flags[rows].groupby(fold_of[rows]).mean()
    return {
        "positives": int(rows.sum()),
        "recall": flags[rows].mean(),
        "round_min": by_round.min(),
        "round_max": by_round.max(),
        "top_1pct": (ranks[rows] >= 0.99).mean(),
        "median_rank": ranks[rows].median(),
    }


def profile(mask):
    return {
        "accounts": int(mask.sum()),
        "median_tx": total_tx[mask].median(),
        "one_labelled_tx": (bad_count[mask] == 1).mean(),
        "median_labelled_share": (bad_count[mask] / total_tx[mask]).median(),
        "any_ach_tx": any_ach[mask].mean(),
        "n_currencies_2plus": (X.loc[mask, "n_currencies"] >= 2).mean(),
        "in_pattern_file": in_pattern[mask].mean(),
    }


print(
    pd.DataFrame(
        {
            "ACH label": profile(positive & has_ach_label),
            "non-ACH label": profile(positive & non_ach_label),
            "tail only": profile(tail_only),
            "all negatives": profile(negative),
        }
    )
    .T.round(3)
    .to_string()
)

count_bins = {
    "1 labelled": bad_count == 1,
    "2-5 labelled": bad_count.between(2, 5),
    "6+ labelled": bad_count >= 6,
}
print("\nby number of labelled transactions")
rows = {}
for count_name, count_mask in count_bins.items():
    rows[("ACH label", count_name)] = recall_row(has_ach_label & count_mask)
    rows[("non-ACH label", count_name)] = recall_row(non_ach_label & count_mask)
print(pd.DataFrame(rows).T.sort_index().round(3).to_string())

print("\none labelled transaction only, by other account traits")
single = bad_count == 1
rows = {}
for label_name, label_mask in {
    "ACH label": has_ach_label,
    "non-ACH label": non_ach_label,
}.items():
    rows[(label_name, "in pattern file")] = recall_row(single & label_mask & in_pattern)
    rows[(label_name, "not in pattern file")] = recall_row(
        single & label_mask & ~in_pattern
    )
    rows[(label_name, "1 currency")] = recall_row(
        single & label_mask & (X["n_currencies"] == 1)
    )
    rows[(label_name, "2+ currencies")] = recall_row(
        single & label_mask & (X["n_currencies"] >= 2)
    )
print(pd.DataFrame(rows).T.round(3).to_string())

               accounts  median_tx  one_labelled_tx  median_labelled_share  any_ach_tx  n_currencies_2plus  in_pattern_file
ACH label        5122.0       18.0            0.807                  0.069       1.000               0.246            0.522
non-ACH label     740.0       33.0            0.957                  0.030       0.462               0.019            0.004
tail only         485.0       17.0            0.000                  0.000       0.575               0.012            1.000
all negatives  508731.0        4.0            0.000                  0.000       0.463               0.015            0.000

by number of labelled transactions


                            positives  recall  round_min  round_max  top_1pct  median_rank
ACH label     1 labelled       4136.0   0.213      0.144      0.271     0.284        0.963
              2-5 labelled      861.0   0.750      0.682      0.787     0.826        0.998
              6+ labelled       125.0   0.928      0.885      1.000     0.960        0.998
non-ACH label 1 labelled        708.0   0.003      0.000      0.008     0.018        0.863
              2-5 labelled       17.0   0.000      0.000      0.000     0.000        0.880
              6+ labelled        15.0   1.000      1.000      1.000     1.000        0.999

one labelled transaction only, by other account traits
                                   positives  recall  round_min  round_max  top_1pct  median_rank
ACH label     in pattern file         1754.0   0.468      0.333      0.544     0.556        0.994
              not in pattern file     2382.0   0.026      0.012      0.065     0.083        0.941
             

How common each payment format is among labelled and other transactions in the window, and the recall of accounts whose labelled transactions all have one format.

In [3]:
window = df[in_window]
by_format = window.groupby("payment_format")["is_laundering"].agg(
    tx="size", labelled="sum"
)
by_format["labelled_rate"] = by_format["labelled"] / by_format["tx"]
by_format["share_of_labelled"] = by_format["labelled"] / by_format["labelled"].sum()

bad_rows = pd.DataFrame(
    {"s": sender[bad], "r": receiver[bad], "f": df.loc[bad, "payment_format"]}
)
bad_formats = pd.concat(
    [
        bad_rows[["s", "f"]].rename(columns={"s": "account"}),
        bad_rows.loc[bad_rows["s"] != bad_rows["r"], ["r", "f"]].rename(
            columns={"r": "account"}
        ),
    ]
).drop_duplicates()
formats_of_account = bad_formats.groupby("account")["f"].agg(["nunique", "first"])
single_format = formats_of_account.loc[formats_of_account["nunique"] == 1, "first"]
only_format = single_format.reindex(X.index)
recall_by_format = pd.DataFrame(
    {name: recall_row(only_format == name) for name in by_format.index}
).T[["positives", "recall", "round_min", "round_max", "median_rank"]]
print(by_format.join(recall_by_format).round(4).to_string())

                     tx  labelled  labelled_rate  share_of_labelled  positives  recall  round_min  round_max  median_rank
payment_format                                                                                                           
ACH              599689      3828         0.0064             0.8465     5109.0  0.3216     0.2849     0.3694       0.9770
Bitcoin          146091        56         0.0004             0.0124       89.0  0.0112     0.0000     0.0312       0.8154
Cash             490891       108         0.0002             0.0239      105.0  0.0000     0.0000     0.0000       0.8678
Cheque          1864331       324         0.0002             0.0716      334.0  0.0060     0.0000     0.0156       0.8708
Credit Card     1323324       206         0.0002             0.0456      192.0  0.0000     0.0000     0.0000       0.8595
Reinvestment     481056         0         0.0000             0.0000        0.0     NaN        NaN        NaN          NaN
Wire             171855 

## 2. Would the model find non-ACH positives if it were trained differently?

Three retrained variants, each scored on the same test folds:

- without the four ACH features;
- without the ACH features and the two currency features;
- with only non-ACH positives in training and validation (other positives are dropped, negatives stay).

If the last variant finds non-ACH positives much better, ACH positives were dominating training. If it does not, these features do not separate them. Each group is scored against all test negatives. `recall_at_budget` flags as many accounts per round as the final model flagged, so no threshold is chosen.

In [4]:
%%time
ACH_COLUMNS = ["ach_out_tx", "ach_in_tx", "ach_receivers", "ach_senders"]
CURRENCY_COLUMNS = ["currency_change_tx", "n_currencies"]
no_ach = [c for c in ALL_FEATURES if c not in ACH_COLUMNS]
no_ach_currency = [c for c in no_ach if c not in CURRENCY_COLUMNS]
drop_from_fit = positive & ~non_ach_label


def fit_on_non_ach(X_train, y_train, X_val, y_val):
    keep_train = ~drop_from_fit.loc[X_train.index]
    keep_val = ~drop_from_fit.loc[X_val.index]
    return fit_final(
        X_train[keep_train], y_train[keep_train], X_val[keep_val], y_val[keep_val]
    )


variant_scores = {
    "final": scores,
    "no ACH features": cv_predict(X[no_ach], y, fold_of, fit_final).scores,
    "no ACH or currency features": cv_predict(
        X[no_ach_currency], y, fold_of, fit_final
    ).scores,
    "trained on non-ACH positives": cv_predict(X, y, fold_of, fit_on_non_ach).scores,
}

target_groups = {
    "all positives": positive,
    "ACH label": positive & has_ach_label,
    "non-ACH label": positive & non_ach_label,
}
rows = []
for name, values in variant_scores.items():
    for group_name, group in target_groups.items():
        for k in range(N_FOLDS):
            test = fold_of == k
            subset = test & (negative | group)
            budget = int(flags[test].sum())
            chosen = values[test].nlargest(budget).index
            rows.append(
                {
                    "variant": name,
                    "group": group_name,
                    "round": k,
                    "auc_pr": average_precision_score(y[subset], values[subset]),
                    "auc_roc": roc_auc_score(y[subset], values[subset]),
                    "recall_at_budget": group.loc[chosen].sum() / (test & group).sum(),
                    "precision_at_budget": y.loc[chosen].mean(),
                }
            )
variant_table = pd.DataFrame(rows)
print(
    variant_table.groupby(["group", "variant"], sort=False)[
        ["auc_pr", "auc_roc", "recall_at_budget", "precision_at_budget"]
    ]
    .agg(["mean", "std"])
    .round(4)
    .to_string()
)
print(
    "\nbase rate of non-ACH positives against negatives:",
    round(
        (positive & non_ach_label).sum()
        / ((positive & non_ach_label).sum() + negative.sum()),
        5,
    ),
)

wide = variant_table.pivot_table(
    index=["group", "round"], columns="variant", values="auc_pr"
)
gains = wide.sub(wide["final"], axis=0).drop(columns="final")
print("\nAUC-PR gain against the final model, per round: mean, sd")
print(gains.groupby("group").agg(["mean", "std"]).round(4).to_string())

                                            auc_pr         auc_roc         recall_at_budget         precision_at_budget        
                                              mean     std    mean     std             mean     std                mean     std
group         variant                                                                                                          
all positives final                         0.3639  0.0153  0.9109  0.0072           0.2620  0.0283              0.7153  0.0888
ACH label     final                         0.4259  0.0147  0.9448  0.0029           0.3211  0.0318              0.7153  0.0888
non-ACH label final                         0.0195  0.0135  0.8607  0.0080           0.0235  0.0180              0.7153  0.0888
all positives no ACH features               0.3275  0.0269  0.8929  0.0050           0.2409  0.0330              0.6564  0.0916
ACH label     no ACH features               0.3816  0.0310  0.9224  0.0045           0.2952  0.0376     

The same question within accounts that have no ACH transaction at all, and the score each group gets against negatives of its own kind.

In [5]:
rows = {}
strata = {
    "non-ACH label, no ACH tx at all, vs negatives with no ACH tx": (
        positive & non_ach_label & ~any_ach,
        negative & ~any_ach,
    ),
    "non-ACH label, some other ACH tx, vs negatives with ACH tx": (
        positive & non_ach_label & any_ach,
        negative & any_ach,
    ),
    "ACH label vs negatives with ACH tx": (
        positive & has_ach_label,
        negative & any_ach,
    ),
}
for name, (pos_mask, neg_mask) in strata.items():
    rounds = []
    for k in range(N_FOLDS):
        subset = (fold_of == k) & (pos_mask | neg_mask)
        rounds.append(roc_auc_score(y[subset], scores[subset]))
    rows[name] = {
        "positives": int(pos_mask.sum()),
        "negatives": int(neg_mask.sum()),
        "base_rate": pos_mask.sum() / (pos_mask.sum() + neg_mask.sum()),
        "auc_roc_mean": np.mean(rounds),
        "auc_roc_sd": np.std(rounds, ddof=1),
        "recall": flags[pos_mask].mean(),
        "flag_rate_negatives": flags[neg_mask].mean(),
    }
print(pd.DataFrame(rows).T.round(4).to_string())

                                                              positives  negatives  base_rate  auc_roc_mean  auc_roc_sd  recall  flag_rate_negatives
non-ACH label, no ACH tx at all, vs negatives with no ACH tx      398.0   273029.0     0.0015        0.9630      0.0058  0.0377                0.000
non-ACH label, some other ACH tx, vs negatives with ACH tx        342.0   235702.0     0.0014        0.7802      0.0131  0.0058                0.003
ACH label vs negatives with ACH tx                               5122.0   235702.0     0.0213        0.8934      0.0052  0.3212                0.003


## 3. What the model uses for each group

Mean out-of-fold SHAP value (log-odds) and median feature value per outcome, for the twelve features with the largest mean absolute SHAP value. Then, per feature, how well that feature alone separates two groups (AUC-ROC; 0.5 is no separation, and values below 0.5 mean the lower value goes with the first group).

In [6]:
true_positive = positive & (flags == 1)
false_negative = positive & (flags == 0)
false_positive = negative & (flags == 1)
true_negative = negative & (flags == 0)
outcomes = {
    "TP": true_positive,
    "FP": false_positive,
    "FN": false_negative,
    "FN non-ACH": false_negative & non_ach_label,
    "TN": true_negative,
}
print({name: int(mask.sum()) for name, mask in outcomes.items()})

top = explained[ALL_FEATURES].abs().mean().nlargest(12).index
print("\nmean SHAP value")
print(
    pd.DataFrame({n: explained.loc[m, top].mean() for n, m in outcomes.items()})
    .round(3)
    .to_string()
)
print("\nmedian feature value")
print(
    pd.DataFrame({n: X.loc[m, top].median() for n, m in outcomes.items()})
    .round(3)
    .to_string()
)
print("\nshare with a nonzero value")
print(
    pd.DataFrame(
        {n: (X.loc[m, top].fillna(0) != 0).mean() for n, m in outcomes.items()}
    )
    .round(3)
    .to_string()
)


def single_feature_auc(first, second):
    values = {}
    for column in ALL_FEATURES:
        a, b = X.loc[first, column].dropna(), X.loc[second, column].dropna()
        target = np.r_[np.ones(len(a)), np.zeros(len(b))]
        values[column] = roc_auc_score(target, np.r_[a.to_numpy(), b.to_numpy()])
    return pd.Series(values)


separation = pd.DataFrame(
    {
        "ACH label vs negatives": single_feature_auc(
            positive & has_ach_label, negative
        ),
        "non-ACH label vs negatives": single_feature_auc(
            positive & non_ach_label, negative
        ),
        "TP vs FP": single_feature_auc(true_positive, false_positive),
        "FN vs TN": single_feature_auc(false_negative, true_negative),
    }
)
print("\nsingle-feature AUC-ROC, sorted by distance from 0.5 for non-ACH positives")
order = (
    (separation["non-ACH label vs negatives"] - 0.5)
    .abs()
    .sort_values(ascending=False)
    .index
)
print(separation.loc[order].round(3).to_string())

print(
    "\nfeature with the largest absolute SHAP value for each account: share per outcome"
)
largest = explained[ALL_FEATURES].abs().idxmax(axis=1)
print(
    pd.DataFrame(
        {
            n: largest[m].value_counts(normalize=True).head(6)
            for n, m in outcomes.items()
        }
    )
    .fillna(0)
    .round(3)
    .to_string()
)

{'TP': 1663, 'FP': 720, 'FN': 4684, 'FN non-ACH': 723, 'TN': 508011}

mean SHAP value
                            TP     FP     FN  FN non-ACH     TN
ach_in_tx                1.382  1.135  0.387      -0.145 -0.268
mean_sender_out_degree   0.138  0.181  0.095       0.902 -0.173
ach_receivers            0.674  0.343  0.023      -0.146 -0.187
log_max_sent_usd         0.298  0.166  0.050      -0.022 -0.142
in_tx                    0.103  0.085  0.036       0.084 -0.197
log_max_received_usd     0.156  0.239 -0.010       0.007 -0.173
median_gap_hours         0.131  0.116  0.011       0.008 -0.175
ach_senders              0.117  0.291  0.101      -0.170 -0.149
ach_out_tx               0.465  0.160  0.035      -0.098 -0.100
max_tx_per_day           0.065  0.029  0.026      -0.057  0.090
mean_receiver_in_degree  0.083  0.062  0.040       0.033 -0.048
n_receivers              0.066  0.046  0.039       0.009 -0.040

median feature value


                            TP      FP      FN  FN non-ACH     TN
ach_in_tx                2.000   2.000   1.000       0.000  0.000
mean_sender_out_degree   4.200   4.500   3.000    2379.667  1.000
ach_receivers            1.000   1.000   1.000       0.000  0.000
log_max_sent_usd         9.494   9.281   8.177       7.805  5.385
in_tx                    9.000  10.000  14.000      24.000  2.000
log_max_received_usd     9.288   9.508   8.463       9.007  4.959
median_gap_hours         2.108   2.162   1.233       0.383  5.983
ach_senders              1.000   1.000   1.000       0.000  0.000
ach_out_tx               2.000   2.000   1.000       0.000  0.000
max_tx_per_day           5.000   5.000   5.000       6.000  2.000
mean_receiver_in_degree  3.083   2.640   2.250       2.000  1.000
n_receivers              3.000   3.000   2.000       2.000  1.000

share with a nonzero value
                            TP     FP     FN  FN non-ACH     TN
ach_in_tx                0.990  0.961  0.596      


single-feature AUC-ROC, sorted by distance from 0.5 for non-ACH positives
                           ACH label vs negatives  non-ACH label vs negatives  TP vs FP  FN vs TN
mean_sender_out_degree                      0.669                       0.899     0.460     0.685
in_tx                                       0.670                       0.815     0.478     0.681
max_tx_per_day                              0.683                       0.782     0.471     0.690
sender_banks                                0.723                       0.769     0.363     0.724
active_days                                 0.638                       0.765     0.487     0.648
log_received_usd                            0.672                       0.765     0.422     0.675
n_senders                                   0.737                       0.763     0.376     0.730
max_tx_per_hour                             0.681                       0.762     0.470     0.682
cheque_tx                                  

## 4. Where the errors are

Precision and recall per segment. Missed positives are then split into exclusive groups, applied in the order shown, so the shares add up to all misses.

In [7]:
def segment_table(segments):
    table = pd.DataFrame(
        {
            "accounts": segments.value_counts(sort=False),
            "positives": y.groupby(segments, observed=True).sum(),
            "flagged": flags.groupby(segments, observed=True).sum(),
            "tp": true_positive.groupby(segments, observed=True).sum(),
            "fp": false_positive.groupby(segments, observed=True).sum(),
            "fn": false_negative.groupby(segments, observed=True).sum(),
        }
    )
    table["positive_rate"] = table["positives"] / table["accounts"]
    table["precision"] = table["tp"] / table["flagged"]
    table["recall"] = table["tp"] / table["positives"]
    table["share_of_fp"] = table["fp"] / table["fp"].sum()
    table["share_of_fn"] = table["fn"] / table["fn"].sum()
    return table.round(3).to_string()


print("transactions in the window")
print(
    segment_table(
        pd.cut(
            total_tx,
            [0, 1, 2, 5, 10, 50, 200, np.inf],
            labels=["1", "2", "3-5", "6-10", "11-50", "51-200", "201+"],
        )
    )
)
print("\ncurrencies used")
print(
    segment_table(
        X["n_currencies"].clip(upper=3).map({0: "0", 1: "1", 2: "2", 3: "3+"})
    )
)
print("\nACH transactions")
ach_total = X["ach_out_tx"] + X["ach_in_tx"]
print(
    segment_table(
        pd.cut(
            ach_total,
            [-1, 0, 1, 3, 10, np.inf],
            labels=["0", "1", "2-3", "4-10", "11+"],
        )
    )
)
print("\ndirection of money")
direction = pd.Series(
    np.select(
        [X["out_tx"] == 0, X["in_tx"] == 0], ["receives only", "sends only"], "both"
    ),
    index=X.index,
)
print(segment_table(direction))
print("\naccounts of the same owner")
owner_of = owners.reindex(X.index)
owner_size = owner_of.map(owner_of.value_counts())
print(
    segment_table(
        pd.cut(
            owner_size,
            [0, 1, 5, 50, 1000, np.inf],
            labels=["1", "2-5", "6-50", "51-1000", "1001+"],
        )
    )
)

share = (bad_count / total_tx).where(positive)
print("\npositives only: share of their transactions that are labelled")
shares = pd.cut(share[share > 0], [0, 0.02, 0.05, 0.1, 0.25, 0.5, 1])
print(
    pd.DataFrame(
        {
            "positives": shares.value_counts(sort=False),
            "recall": flags[shares.index].groupby(shares, observed=True).mean(),
            "median_rank": ranks[shares.index].groupby(shares, observed=True).median(),
        }
    )
    .round(3)
    .to_string()
)

conditions = [
    tail_only,
    non_ach_label,
    has_ach_label & (bad_count == 1) & (X["n_currencies"] == 1),
    has_ach_label & (bad_count == 1),
    has_ach_label,
]
names = [
    "no labelled tx in the window",
    "labelled tx, none ACH",
    "one ACH labelled tx, one currency",
    "one ACH labelled tx, 2+ currencies",
    "2+ labelled tx, some ACH",
]
kind = pd.Series(np.select(conditions, names, "other"), index=X.index)[positive]
misses = (
    pd.DataFrame(
        {
            "positives": kind.value_counts(),
            "missed": (flags[positive] == 0).groupby(kind).sum(),
            "recall": flags[positive].groupby(kind).mean(),
            "median_rank": ranks[positive].groupby(kind).median(),
        }
    )
    .reindex([*names, "other"])
    .fillna(0)
)
misses["share_of_misses"] = misses["missed"] / misses["missed"].sum()
print("\nmissed positives, exclusive groups")
print(misses.round(3).to_string())
assert misses["positives"].sum() == positive.sum()

transactions in the window
        accounts  positives  flagged   tp   fp    fn  positive_rate  precision  recall  share_of_fp  share_of_fn
count                                                                                                           
1          90862        109       10    8    2   101          0.001      0.800   0.073        0.003        0.022
2         106503        267       80   49   31   218          0.003      0.612   0.184        0.043        0.047
3-5        78240       1103      496  380  116   723          0.014      0.766   0.345        0.161        0.154
6-10       25562        825      352  242  110   583          0.032      0.688   0.293        0.153        0.124
11-50     164254       2965      942  637  305  2328          0.018      0.676   0.215        0.424        0.497
51-200     49407       1042      476  324  152   718          0.021      0.681   0.311        0.211        0.153
201+         250         36       27   23    4    13          0.144  

              accounts  positives  flagged    tp   fp    fn  positive_rate  precision  recall  share_of_fp  share_of_fn
n_currencies                                                                                                           
1               506013       5069     1053   480  573  4589          0.010      0.456   0.095        0.796        0.980
2                 8741       1198     1251  1112  139    86          0.137      0.889   0.928        0.193        0.018
3+                 324         80       79    71    8     9          0.247      0.899   0.888        0.011        0.002

ACH transactions


      accounts  positives  flagged   tp   fp    fn  positive_rate  precision  recall  share_of_fp  share_of_fn
0       273633        604       28   15   13   589          0.002      0.536   0.025        0.018        0.126
1        88842       1244      195  108   87  1136          0.014      0.554   0.087        0.121        0.243
2-3      70517       1636      583  399  184  1237          0.023      0.684   0.244        0.256        0.264
4-10     54594       1788      997  747  250  1041          0.033      0.749   0.418        0.347        0.222
11+      27492       1075      580  394  186   681          0.039      0.679   0.367        0.258        0.145

direction of money


               accounts  positives  flagged    tp   fp    fn  positive_rate  precision  recall  share_of_fp  share_of_fn
both             402520       6080     2349  1647  702  4433          0.015      0.701   0.271        0.975        0.946
receives only     18105        176       32    16   16   160          0.010      0.500   0.091        0.022        0.034
sends only        94453         91        2     0    2    91          0.001      0.000   0.000        0.003        0.019

accounts of the same owner


           accounts  positives  flagged   tp   fp    fn  positive_rate  precision  recall  share_of_fp  share_of_fn
entity_id                                                                                                          
1            102971       2085      625  439  186  1646          0.020      0.702   0.211        0.258        0.351
2-5          147969       1732      641  451  190  1281          0.012      0.704   0.260        0.264        0.273
6-50         149001       1552      697  498  199  1054          0.010      0.714   0.321        0.276        0.225
51-1000       53122        491      255  171   84   320          0.009      0.671   0.348        0.117        0.068
1001+         62015        487      165  104   61   383          0.008      0.630   0.214        0.085        0.082

positives only: share of their transactions that are labelled
              positives  recall  median_rank
count                                       
(0.0, 0.02]         835   0.240    

False positives against true positives and against all negatives: how close they sit to positive accounts in the graph. `transfer with a flagged positive` means a counterparty that is itself a positive the model flagged.

In [8]:
pairs = pd.DataFrame(
    {"a": sender[in_window & ~is_self], "b": receiver[in_window & ~is_self]}
).drop_duplicates()
both_ways = pd.concat(
    [pairs, pairs.rename(columns={"a": "b", "b": "a"})]
).drop_duplicates()


def counterparties_in(accounts):
    found = both_ways[both_ways["b"].isin(accounts)].groupby("a").size()
    return found.reindex(X.index, fill_value=0)


n_positive_nb = counterparties_in(set(y.index[positive]))
n_flagged_nb = counterparties_in(set(y.index[true_positive]))
n_counterparties = both_ways.groupby("a").size().reindex(X.index, fill_value=0)

print(
    pd.DataFrame(
        {
            name: {
                "accounts": int(mask.sum()),
                "any positive counterparty": (n_positive_nb[mask] > 0).mean(),
                "transfer with a flagged positive": (n_flagged_nb[mask] > 0).mean(),
                "2+ positive counterparties": (n_positive_nb[mask] >= 2).mean(),
                "median counterparties": n_counterparties[mask].median(),
                "median positive share of counterparties": (
                    n_positive_nb[mask] / n_counterparties[mask].clip(lower=1)
                ).median(),
            }
            for name, mask in outcomes.items()
        }
    )
    .T.round(3)
    .to_string()
)

print("\nflagged accounts by number of positive counterparties")
print(segment_table(n_positive_nb.clip(upper=3).map({0: "0", 1: "1", 2: "2", 3: "3+"})))

print("\nscore bands among flagged accounts")
bands = pd.cut(scores[flags == 1], [0, 0.3, 0.5, 0.7, 0.9, 1])
print(
    pd.DataFrame(
        {
            "flagged": bands.value_counts(sort=False),
            "false_positives": false_positive[bands.index]
            .groupby(bands, observed=True)
            .sum(),
            "precision": y[bands.index].groupby(bands, observed=True).mean(),
        }
    )
    .round(3)
    .to_string()
)

            accounts  any positive counterparty  transfer with a flagged positive  2+ positive counterparties  median counterparties  median positive share of counterparties
TP            1663.0                      1.000                             0.826                       0.544                    4.0                                    0.500
FP             720.0                      0.275                             0.211                       0.040                    5.0                                    0.000
FN            4684.0                      0.912                             0.392                       0.145                    3.0                                    0.333
FN non-ACH     723.0                      1.000                             0.859                       0.048                    3.0                                    0.333
TN          508011.0                      0.113                             0.084                       0.005                    2

## 5. Are the same accounts wrong under other fold assignments?

The final model again on fold seeds 1 and 2, with the same threshold rule. An account that stays wrong when it is scored by models trained on different data is a systematic error. One that changes is closer to chance.

In [9]:
%%time


def best_f1_threshold(y_val, val_scores):
    precision, recall, thresholds = precision_recall_curve(y_val, val_scores)
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return thresholds[f1.argmax()]


runs = {0: results[["score", "flag"]]}
for fold_seed in [1, 2]:
    other_folds = cached_folds(labels, groups, DATA_INTERIM, seed=fold_seed).loc[
        X.index
    ]
    other = cv_predict(X, y, other_folds, fit_final, best_f1_threshold)
    runs[fold_seed] = pd.DataFrame({"score": other.scores, "flag": other.flags})
    rounds = [
        average_precision_score(y[other_folds == k], other.scores[other_folds == k])
        for k in range(N_FOLDS)
    ]
    print(
        f"fold seed {fold_seed}: AUC-PR {np.mean(rounds):.4f} "
        f"+/- {np.std(rounds, ddof=1):.4f}, flagged {int(other.flags.sum())}"
    )

all_flags = pd.DataFrame({seed: run["flag"] for seed, run in runs.items()})
times_flagged = all_flags.sum(axis=1)
print("\nnumber of the 3 runs in which an account is flagged")
print(
    pd.DataFrame(
        {
            "negatives": times_flagged[negative].value_counts().sort_index(),
            "positives": times_flagged[positive].value_counts().sort_index(),
        }
    )
    .fillna(0)
    .astype(int)
    .to_string()
)

print("\nof the accounts wrong in the original run, share also wrong in another run")
rows = {}
for seed in [1, 2]:
    rows[f"fold seed {seed}"] = {
        "FP stays flagged": all_flags.loc[false_positive, seed].mean(),
        "FN stays missed": 1 - all_flags.loc[false_negative, seed].mean(),
        "TP stays flagged": all_flags.loc[true_positive, seed].mean(),
    }
print(pd.DataFrame(rows).round(3).to_string())

score_table = pd.DataFrame({seed: run["score"] for seed, run in runs.items()})
print("\nSpearman correlation of scores between runs: negatives, then positives")
print(score_table[negative].corr(method="spearman").round(3).to_string())
print(score_table[positive].corr(method="spearman").round(3).to_string())

print("\nprofile by how often a false positive recurs")
always_fp = negative & (times_flagged == 3)
once_fp = negative & (times_flagged == 1)
print(
    pd.DataFrame(
        {
            name: {
                "accounts": int(mask.sum()),
                "any positive counterparty": (n_positive_nb[mask] > 0).mean(),
                "n_currencies_2plus": (X.loc[mask, "n_currencies"] >= 2).mean(),
                "any_ach_tx": any_ach[mask].mean(),
                "median_tx": total_tx[mask].median(),
                "median_score_original_run": scores[mask].median(),
            }
            for name, mask in {
                "flagged in all 3": always_fp,
                "flagged in 1": once_fp,
            }.items()
        }
    )
    .T.round(3)
    .to_string()
)

fold seed 1: AUC-PR 0.3603 +/- 0.0146, flagged 2749


fold seed 2: AUC-PR 0.3634 +/- 0.0154, flagged 2588

number of the 3 runs in which an account is flagged
   negatives  positives
0     507114       4513
1        911        168
2        360        115
3        346       1551

of the accounts wrong in the original run, share also wrong in another run
                  fold seed 1  fold seed 2
FP stays flagged        0.651        0.568
FN stays missed         0.977        0.977
TP stays flagged        0.957        0.952

Spearman correlation of scores between runs: negatives, then positives


       0      1      2
0  1.000  0.974  0.974
1  0.974  1.000  0.976
2  0.974  0.976  1.000
       0      1      2
0  1.000  0.966  0.966
1  0.966  1.000  0.965
2  0.966  0.965  1.000

profile by how often a false positive recurs
                  accounts  any positive counterparty  n_currencies_2plus  any_ach_tx  median_tx  median_score_original_run
flagged in all 3     346.0                       0.24               0.338       0.997       17.0                      0.373
flagged in 1         911.0                       0.31               0.067       0.964       25.0                      0.147
CPU times: total: 43min 3s
Wall time: 2min 55s


## 6. Why does a currency change go with a positive label?

No labelled transaction changes currency, yet many accounts with exactly one change are positive. For those accounts, this compares the change with the labelled transaction: the order in time, the gap, the currencies, and the payment formats.

In [10]:
changes = in_window & (df["payment_currency"] != df["receiving_currency"])
print("labelled transactions that change currency:", int((changes & bad).sum()))
print("payment formats of currency changes:")
print(df.loc[changes, "payment_format"].value_counts().to_string())
print("self-transaction share of currency changes:", round(is_self[changes].mean(), 4))

one_change = X["currency_change_tx"] == 1
print("\nexactly one change, by outcome")
print(segment_table(one_change.map({True: "one change", False: "other"})))

chosen = one_change & positive & (bad_count > 0)
accounts = set(X.index[chosen])
change_rows = pd.DataFrame(
    {
        "account": sender[changes],
        "change_time": df.loc[changes, "timestamp"],
        "paid": df.loc[changes, "payment_currency"],
        "received": df.loc[changes, "receiving_currency"],
    }
)
change_rows = change_rows[change_rows["account"].isin(accounts)]
bad_touch = pd.concat(
    [
        pd.DataFrame({"account": sender[bad], "role": "sent"}),
        pd.DataFrame({"account": receiver[bad & ~is_self], "role": "received"}),
    ]
).join(df.loc[bad, ["timestamp", "payment_currency", "payment_format"]])
joined = change_rows.merge(bad_touch[bad_touch["account"].isin(accounts)], on="account")
joined["hours_after_change"] = (
    joined["timestamp"] - joined["change_time"]
).dt.total_seconds() / 3600
first = joined.loc[
    joined["hours_after_change"].abs().groupby(joined["account"]).idxmin()
]
print("\npositives with one change and a labelled tx:", len(first))
print("labelled tx closest to the change")
print(
    pd.Series(
        {
            "labelled tx after the change": (first["hours_after_change"] > 0).mean(),
            "median absolute gap, hours": first["hours_after_change"].abs().median(),
            "within 1 hour": (first["hours_after_change"].abs() <= 1).mean(),
            "within 24 hours": (first["hours_after_change"].abs() <= 24).mean(),
            "labelled currency = paid side of change": (
                first["payment_currency"] == first["paid"]
            ).mean(),
            "labelled currency = received side of change": (
                first["payment_currency"] == first["received"]
            ).mean(),
            "labelled tx is ACH": (first["payment_format"] == "ACH").mean(),
            "account sent the labelled tx": (first["role"] == "sent").mean(),
        }
    )
    .round(3)
    .to_string()
)
print("\nchange direction")
print(
    pd.Series(
        {
            "US Dollar on the received side": (first["received"] == "US Dollar").mean(),
            "US Dollar on the paid side": (first["paid"] == "US Dollar").mean(),
        }
    )
    .round(3)
    .to_string()
)
print(
    "\ncurrency of labelled transactions in the window, against all window transactions"
)
print(
    pd.DataFrame(
        {
            "labelled": df.loc[bad, "payment_currency"].value_counts(normalize=True),
            "all": df.loc[in_window, "payment_currency"].value_counts(normalize=True),
        }
    )
    .sort_values("labelled", ascending=False)
    .head(6)
    .round(3)
    .to_string()
)

labelled transactions that change currency: 0
payment formats of currency changes:
payment_format
ACH     71191
Wire      588
self-transaction share of currency changes: 0.9695

exactly one change, by outcome


                    accounts  positives  flagged    tp   fp    fn  positive_rate  precision  recall  share_of_fp  share_of_fn
currency_change_tx                                                                                                           
one change              2860       1105     1206  1092  114    13          0.386      0.905   0.988        0.158        0.003
other                 512218       5242     1177   571  606  4671          0.010      0.485   0.109        0.842        0.997



positives with one change and a labelled tx: 1104
labelled tx closest to the change
labelled tx after the change                   0.005
median absolute gap, hours                     0.000
within 1 hour                                  0.995
within 24 hours                                0.995
labelled currency = paid side of change        0.005
labelled currency = received side of change    0.995
labelled tx is ACH                             0.999
account sent the labelled tx                   0.996

change direction
US Dollar on the received side    0.359
US Dollar on the paid side        0.311

currency of labelled transactions in the window, against all window transactions


                  labelled    all
payment_currency                 
US Dollar            0.369  0.373
Euro                 0.268  0.230
Saudi Riyal          0.067  0.018
Yuan                 0.037  0.042
Swiss Franc          0.036  0.046
Rupee                0.033  0.037


## 7. Follow-ups

### 7.1 Who are the non-ACH positives?

Their largest feature is the out-degree of their senders, whose median is very high. This looks at who sends the labelled non-ACH transactions, and how rare a positive is among the other accounts paid by the same senders.

In [11]:
non_ach_tx = bad & ~is_ach
out_degree = pairs.groupby("a").size()
label_senders = sender[non_ach_tx]
label_receivers = receiver[non_ach_tx]
sender_degree = label_senders.map(out_degree).fillna(0)
sender_counts = label_senders.value_counts()
group = positive & non_ach_label
received_any = X.index.isin(label_receivers)
sent_any = X.index.isin(label_senders)
print(
    pd.Series(
        {
            "labelled non-ACH transactions": len(label_senders),
            "distinct senders": label_senders.nunique(),
            "distinct receivers": label_receivers.nunique(),
            "share from the top sender": sender_counts.iloc[0] / len(label_senders),
            "share from the top 10 senders": sender_counts.head(10).sum()
            / len(label_senders),
            "median out-degree of the sender": sender_degree.median(),
            "share with sender out-degree 100+": (sender_degree >= 100).mean(),
            "non-ACH positives that received one": (group & received_any).sum(),
            "non-ACH positives that sent one": (group & sent_any).sum(),
        }
    )
    .round(3)
    .to_string()
)

hubs = sender_counts.index[sender_counts.index.map(out_degree).fillna(0) >= 100]
hub_links = pairs[pairs["a"].isin(hubs)]
hub_tx = in_window & ~is_self & sender.isin(hubs)
hub_receivers = pd.Index(hub_links["b"].unique()).intersection(X.index)
print(f"\nsenders with out-degree 100+: {len(hubs)}")
print(
    pd.Series(
        {
            "flagged share of these senders": flags.reindex(hubs).mean(),
            "median labelled tx per sender": sender_counts[hubs].median(),
            "labelled share of their outgoing tx": df.loc[
                hub_tx, "is_laundering"
            ].mean(),
            "accounts they pay": len(hub_receivers),
            "positive share of the accounts they pay": y[hub_receivers].mean(),
            "share of all non-ACH positives they pay": group[hub_receivers].sum()
            / group.sum(),
        }
    )
    .round(4)
    .to_string()
)
print("\nby mean out-degree of an account's senders")
print(
    segment_table(
        pd.cut(
            X["mean_sender_out_degree"],
            [-1, 0, 5, 20, 100, 1000, np.inf],
            labels=["0", "1-5", "6-20", "21-100", "101-1000", "1001+"],
        )
    )
)

labelled non-ACH transactions           694.000
distinct senders                         73.000
distinct receivers                      680.000
share from the top sender                 0.350
share from the top 10 senders             0.823
median out-degree of the sender        8846.000
share with sender out-degree 100+         0.912
non-ACH positives that received one     669.000
non-ACH positives that sent one          71.000



senders with out-degree 100+: 15
flagged share of these senders                 1.0000
median labelled tx per sender                 18.0000
labelled share of their outgoing tx            0.0014
accounts they pay                          38254.0000
positive share of the accounts they pay        0.0288
share of all non-ACH positives they pay        0.8311

by mean out-degree of an account's senders
                        accounts  positives  flagged   tp   fp    fn  positive_rate  precision  recall  share_of_fp  share_of_fn
mean_sender_out_degree                                                                                                          
0                         230779        767      205  175   30   592          0.003      0.854   0.228        0.042        0.126
1-5                       175296       3109     1142  777  365  2332          0.018      0.680   0.250        0.507        0.498
6-20                       70412       1363      742  545  197   818          0.01

### 7.2 Positives with one labelled ACH transaction and one currency

This group holds most of the missed positives. Here they are split by the pattern file and by the role of the account in the labelled transaction, and each feature alone is compared with negatives that also have an ACH transaction.

In [12]:
lane = positive & has_ach_label & (bad_count == 1) & (X["n_currencies"] == 1)
ach_bad = bad & is_ach
sent_ach_label = pd.Series(X.index.isin(sender[ach_bad]), index=X.index)
rows = {
    "all": recall_row(lane),
    "in pattern file": recall_row(lane & in_pattern),
    "not in pattern file": recall_row(lane & ~in_pattern),
    "sent the labelled tx": recall_row(lane & sent_ach_label),
    "received the labelled tx": recall_row(lane & ~sent_ach_label),
    "in pattern, sent": recall_row(lane & in_pattern & sent_ach_label),
    "in pattern, received": recall_row(lane & in_pattern & ~sent_ach_label),
    "not in pattern, sent": recall_row(lane & ~in_pattern & sent_ach_label),
    "not in pattern, received": recall_row(lane & ~in_pattern & ~sent_ach_label),
}
print(pd.DataFrame(rows).T.round(3).to_string())

print("\nprofile against other groups")
print(
    pd.DataFrame(
        {
            "this group, missed": profile(lane & (flags == 0)),
            "this group, flagged": profile(lane & (flags == 1)),
            "negatives with ACH tx": profile(negative & any_ach),
        }
    )
    .T.round(3)
    .to_string()
)
print(
    pd.DataFrame(
        {
            "this group, missed": X[lane & (flags == 0)][top].median(),
            "this group, flagged": X[lane & (flags == 1)][top].median(),
            "negatives with ACH tx": X[negative & any_ach][top].median(),
        }
    )
    .round(3)
    .to_string()
)

lane_auc = single_feature_auc(lane, negative & any_ach)
order = (lane_auc - 0.5).abs().sort_values(ascending=False).index[:8]
print("\nsingle-feature AUC-ROC against negatives with an ACH tx")
print(lane_auc[order].round(3).to_string())
rounds = []
for k in range(N_FOLDS):
    subset = (fold_of == k) & (lane | (negative & any_ach))
    rounds.append(roc_auc_score(y[subset], scores[subset]))
print(
    f"model AUC-ROC on the same comparison: {np.mean(rounds):.4f} "
    f"+/- {np.std(rounds, ddof=1):.4f}"
)

                          positives  recall  round_min  round_max  top_1pct  median_rank
all                          3457.0   0.078      0.051      0.133     0.157        0.950
in pattern file              1107.0   0.193      0.149      0.283     0.321        0.969
not in pattern file          2350.0   0.023      0.010      0.060     0.080        0.941
sent the labelled tx         1427.0   0.018      0.007      0.040     0.060        0.895
received the labelled tx     2030.0   0.120      0.082      0.192     0.225        0.969
in pattern, sent              263.0   0.027      0.000      0.073     0.099        0.908
in pattern, received          844.0   0.245      0.203      0.328     0.390        0.980
not in pattern, sent         1164.0   0.015      0.009      0.035     0.051        0.892
not in pattern, received     1186.0   0.030      0.008      0.084     0.108        0.966

profile against other groups
                       accounts  median_tx  one_labelled_tx  median_labelled_sha


single-feature AUC-ROC against negatives with an ACH tx
ach_senders                0.669
n_senders                  0.652
sender_banks               0.645
log_median_received_usd    0.621
log_max_received_usd       0.598
ach_in_tx                  0.594
mean_sender_out_degree     0.581
flow_balance               0.580
model AUC-ROC on the same comparison: 0.8490 +/- 0.0065


### 7.3 Currency conversions and their same-minute sends

Section 6 shows the labelled transaction of one-change accounts is usually the send that follows the conversion. This relates that to the number of conversions of the account. A conversion is a currency-changing self-transaction; its send has the same account, the same minute and the received currency.

In [13]:
conversions = in_window & is_self & (df["payment_currency"] != df["receiving_currency"])
keys = pd.DataFrame(
    {
        "account": sender[conversions],
        "timestamp": df.loc[conversions, "timestamp"],
        "payment_currency": df.loc[conversions, "receiving_currency"],
    }
).drop_duplicates()
sends = pd.DataFrame(
    {
        "account": sender[in_window & ~is_self],
        "timestamp": df.loc[in_window & ~is_self, "timestamp"],
        "payment_currency": df.loc[in_window & ~is_self, "payment_currency"],
        "is_laundering": df.loc[in_window & ~is_self, "is_laundering"],
    }
)
paired = sends.merge(keys, on=["account", "timestamp", "payment_currency"])
per_account = sender[conversions].value_counts().reindex(X.index, fill_value=0)
bins = pd.cut(
    per_account,
    [-1, 0, 1, 2, 5, 20, np.inf],
    labels=["0", "1", "2", "3-5", "6-20", "21+"],
)
print("accounts by number of conversions")
print(segment_table(bins))
sends_of = paired.assign(bin=paired["account"].map(bins)).groupby("bin", observed=True)
print("\nsame-minute sends after a conversion, by conversions of the sender")
print(
    pd.DataFrame(
        {
            "sends": sends_of.size(),
            "labelled": sends_of["is_laundering"].sum(),
            "labelled_share": sends_of["is_laundering"].mean(),
        }
    )
    .round(4)
    .to_string()
)
print(
    "\nlabelled ACH sends in the window that follow a conversion:",
    int(paired["is_laundering"].sum()),
)
print("all labelled ACH sends in the window:", int((bad & is_ach & ~is_self).sum()))

one = per_account == 1
one_auc = single_feature_auc(one & positive, one & negative)
order = (one_auc - 0.5).abs().sort_values(ascending=False).index[:8]
print("\nwithin accounts with one conversion: single-feature AUC-ROC, by label")
print(one_auc[order].round(3).to_string())
print(f"model AUC-ROC within them: {roc_auc_score(y[one], scores[one]):.4f}")
print(
    pd.DataFrame(
        {
            "positive": X.loc[one & positive, order].median(),
            "negative": X.loc[one & negative, order].median(),
        }
    )
    .round(3)
    .to_string()
)

accounts by number of conversions
       accounts  positives  flagged    tp   fp    fn  positive_rate  precision  recall  share_of_fp  share_of_fn
count                                                                                                           
0        506513       5072     1056   481  575  4591          0.010      0.455   0.095        0.799        0.980
1          2745       1104     1203  1092  111    12          0.402      0.908   0.989        0.154        0.003
2          1344         24       29    17   12     7          0.018      0.586   0.708        0.017        0.001
3-5         945         38       33    24    9    14          0.040      0.727   0.632        0.012        0.003
6-20       2728         87       59    47   12    40          0.032      0.797   0.540        0.017        0.009
21+         803         22        3     2    1    20          0.027      0.667   0.091        0.001        0.004

same-minute sends after a conversion, by conversions of the s


within accounts with one conversion: single-feature AUC-ROC, by label
ach_receivers          0.954
ach_out_tx             0.949
self_tx                0.875
reinvestment_tx        0.875
receiver_banks         0.127
n_receivers            0.130
in_tx                  0.869
log_median_sent_usd    0.820
model AUC-ROC within them: 0.9930
                     positive  negative
ach_receivers           1.000     0.000
ach_out_tx              2.000     1.000
self_tx                 2.000     1.000
reinvestment_tx         1.000     0.000
receiver_banks          2.000     8.000
n_receivers             3.000     9.000
in_tx                  12.000     2.000
log_median_sent_usd     8.521     4.956


### 7.4 Does removing the features remove the signal?

A conversion is also a self-transaction, so `self_tx` may carry what the currency features carry. Here the final model is retrained without the currency features, and without them and `self_tx`, with and without the ACH features. `recall_at_budget` again uses the number of accounts the final model flagged in that round.

In [14]:
%%time
SELF = ["self_tx"]
no_currency = [c for c in ALL_FEATURES if c not in CURRENCY_COLUMNS]
no_currency_self = [c for c in no_currency if c not in SELF]
no_ach_currency_self = [c for c in no_ach_currency if c not in SELF]
removal_scores = {
    "final": scores,
    "no currency": cv_predict(X[no_currency], y, fold_of, fit_final).scores,
    "no currency, self_tx": cv_predict(
        X[no_currency_self], y, fold_of, fit_final
    ).scores,
    "no ACH, currency": variant_scores["no ACH or currency features"],
    "no ACH, currency, self_tx": cv_predict(
        X[no_ach_currency_self], y, fold_of, fit_final
    ).scores,
}
recall_groups = {
    "all positives": positive,
    "2+ currencies": positive & (X["n_currencies"] >= 2),
    "1 currency": positive & (X["n_currencies"] == 1),
    "one ACH label, one currency": lane,
    "non-ACH label": group,
}
rows = []
for name, values in removal_scores.items():
    for k in range(N_FOLDS):
        test = fold_of == k
        chosen = values[test].nlargest(int(flags[test].sum())).index
        row = {
            "variant": name,
            "round": k,
            "auc_pr": average_precision_score(y[test], values[test]),
            "precision": y[chosen].mean(),
        }
        for group_name, members_of_group in recall_groups.items():
            row[group_name] = (
                members_of_group[chosen].sum() / (test & members_of_group).sum()
            )
        rows.append(row)
removal = pd.DataFrame(rows)
print(
    removal.drop(columns="round")
    .groupby("variant", sort=False)
    .mean()
    .round(4)
    .to_string()
)
wide = removal.pivot(index="round", columns="variant", values="auc_pr")
gains = wide.sub(wide["final"], axis=0).drop(columns="final")
print("\nAUC-PR gain against the final model, per round")
print(gains.agg(["mean", "std"]).T.round(4).to_string())

                           auc_pr  precision  all positives  2+ currencies  1 currency  one ACH label, one currency  non-ACH label
variant                                                                                                                           
final                      0.3639     0.7153         0.2620         0.9246      0.0947                       0.0771         0.0235
no currency                0.3573     0.6939         0.2544         0.8823      0.0956                       0.0762         0.0249
no currency, self_tx       0.3300     0.6356         0.2340         0.7436      0.1043                       0.0832         0.0263
no ACH, currency           0.3224     0.6485         0.2384         0.8819      0.0758                       0.0652         0.0247
no ACH, currency, self_tx  0.2420     0.4685         0.1736         0.4931      0.0927                       0.0759         0.0317

AUC-PR gain against the final model, per round
                             mean  

### 7.5 One conversion, and an ACH send in the same minute

The sends after a conversion differ sharply by payment format. This section measures that split, and then a one-line rule built from it, which uses no label and no model: the account made exactly one conversion in the window, and an ACH send follows it in the same minute. It then scores the final model on the accounts outside the rule, and retrains it with the rule's accounts removed from training and validation.

In [15]:
sends_with_format = sends.assign(
    payment_format=df.loc[sends.index, "payment_format"]
).merge(keys, on=["account", "timestamp", "payment_currency"])
print("same-minute sends after a conversion, by payment format")
print(
    sends_with_format.groupby("payment_format")["is_laundering"]
    .agg(["size", "sum", "mean"])
    .round(4)
    .to_string()
)
window_ach_sends = sends[is_ach[sends.index]]
after_conversion = sends_with_format[sends_with_format["payment_format"] == "ACH"]
rest_size = len(window_ach_sends) - len(after_conversion)
rest_labelled = (
    window_ach_sends["is_laundering"].sum() - after_conversion["is_laundering"].sum()
)
print(f"\nACH sends in the window: {len(window_ach_sends):,}")
print(
    f"after a conversion: {len(after_conversion):,}, "
    f"labelled {after_conversion['is_laundering'].mean():.4f}"
)
print(f"other ACH sends: {rest_size:,}, labelled {rest_labelled / rest_size:.5f}")

one_conversion = set(per_account.index[per_account == 1])
rule_accounts = set(after_conversion["account"]) & one_conversion
rule = pd.Series(X.index.isin(rule_accounts), index=X.index)
outcome = pd.Series(
    np.select(
        [rule, per_account == 1],
        ["one conversion, ACH send", "one conversion, no ACH send"],
        "all other accounts",
    ),
    index=X.index,
)
print()
print(segment_table(outcome))

rows = []
for k in range(N_FOLDS):
    test = fold_of == k
    rows.append(
        {
            "round": k,
            "rule_accounts": rule[test].sum(),
            "rule_precision": y[test & rule].mean(),
            "rule_recall": (test & rule & positive).sum() / (test & positive).sum(),
        }
    )
print()
print(pd.DataFrame(rows).round(3).to_string(index=False))
print(
    "share of the model's true positives inside the rule:",
    round((true_positive & rule).sum() / true_positive.sum(), 3),
)

same-minute sends after a conversion, by payment format
                 size   sum    mean
payment_format                     
ACH              8826  1619  0.1834
Cash             7312     0  0.0000
Cheque          29908     0  0.0000
Credit Card     20519     0  0.0000
Wire             3067     0  0.0000

ACH sends in the window: 528,735
after a conversion: 8,826, labelled 0.1834
other ACH sends: 519,909, labelled 0.00423



                             accounts  positives  flagged    tp   fp    fn  positive_rate  precision  recall  share_of_fp  share_of_fn
all other accounts             512333       5243     1180   571  609  4672          0.010      0.484   0.109        0.846        0.997
one conversion, ACH send         1187       1097     1113  1088   25     9          0.924      0.978   0.992        0.035        0.002
one conversion, no ACH send      1558          7       90     4   86     3          0.004      0.044   0.571        0.119        0.001

 round  rule_accounts  rule_precision  rule_recall
     0            247           0.919        0.179
     1            177           0.944        0.132
     2            253           0.925        0.184
     3            261           0.927        0.191
     4            249           0.912        0.179
share of the model's true positives inside the rule: 0.654


In [16]:
%%time


def fit_without_rule(X_train, y_train, X_val, y_val):
    keep_train = ~rule.loc[X_train.index]
    keep_val = ~rule.loc[X_val.index]
    return fit_final(
        X_train[keep_train], y_train[keep_train], X_val[keep_val], y_val[keep_val]
    )


trained_outside = cv_predict(X, y, fold_of, fit_without_rule).scores
trained_outside_nofeat = cv_predict(
    X[no_ach_currency_self], y, fold_of, fit_without_rule
).scores
rows = []
for k in range(N_FOLDS):
    test = fold_of == k
    outside = test & ~rule
    budget = int(flags[outside].sum())
    row = {"round": k, "base_rate": y[outside].mean(), "flagged": budget}
    for name, values in {
        "final": scores,
        "retrained outside the rule": trained_outside,
        "retrained, no ACH, currency, self_tx": trained_outside_nofeat,
    }.items():
        chosen = values[outside].nlargest(budget).index
        row[f"{name}: auc_pr"] = average_precision_score(y[outside], values[outside])
        row[f"{name}: auc_roc"] = roc_auc_score(y[outside], values[outside])
        row[f"{name}: precision"] = y[chosen].mean()
        row[f"{name}: recall"] = y[chosen].sum() / y[outside].sum()
    rows.append(row)
outside_table = pd.DataFrame(rows).drop(columns="round")
print(outside_table.agg(["mean", "std"]).T.round(4).to_string())

                                                     mean       std
base_rate                                          0.0102    0.0003
flagged                                          254.0000  116.8888
final: auc_pr                                      0.1720    0.0231
final: auc_roc                                     0.8923    0.0103
final: precision                                   0.4756    0.0689
final: recall                                      0.1094    0.0293
retrained outside the rule: auc_pr                 0.1820    0.0203
retrained outside the rule: auc_roc                0.8928    0.0102
retrained outside the rule: precision              0.4695    0.0756
retrained outside the rule: recall                 0.1074    0.0270
retrained, no ACH, currency, self_tx: auc_pr       0.1426    0.0124
retrained, no ACH, currency, self_tx: auc_roc      0.8703    0.0077
retrained, no ACH, currency, self_tx: precision    0.3871    0.0671
retrained, no ACH, currency, self_tx: recall    